# PDA-1. Предобработка текстов с помощью Python

## 8. Задачи машинного обучения на текстах: практика

Импорт библиотек:

In [1]:
import os
import sys
import string
import annoy
import codecs
import setuptools
import pkg_resources

from pymorphy2 import MorphAnalyzer
from stop_words import get_stop_words
from gensim.models import Word2Vec

import numpy as np
from tqdm.notebook import tqdm
import pandas as pd

import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

C:\Users\tgorbunov\AppData\Local\Temp\ipykernel_11784\118036563.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Предобработаем ответы из файла chatbot/Otvety.txt: к каждому вопросу присоединим 1 ответ и запишем в файл на будущее. Это позволит нам сэкономить время и ресурсы при дальнейшем препроцессинге текста

In [2]:
question = None
written = False

#Мы идем по всем записям, берем первую строку как вопрос
# и после знака --- находим ответ
with codecs.open("chatbot/prepared_answers.txt","w", "utf-8") as fout:
    with codecs.open("chatbot/Otvety.txt", "r", "utf-8") as fin:
        for line in tqdm(fin):
            if line.startswith("---"):
                written = False
                continue
            if not written and question is not None:
                fout.write(question.replace("\t", " ").strip() + "\t" + line.replace("\t", " "))
                written = True
                question = None
                continue
            if not written:
                question = line.strip()
                continue

0it [00:00, ?it/s]

Теперь нам нужно предобработать текст, чтобы обучить word2vec и получить эмбеддинги. Удаляем знаки препинания и делаем лемматизацию

In [3]:
def preprocess_txt(line):
    spls = "".join(i for i in line.strip() if i not in exclude).split()
    spls = [morpher.parse(i.lower())[0].normal_form for i in spls]
    spls = [i for i in spls if i not in sw and i != ""]
    return spls

In [4]:
sentences = []

morpher = MorphAnalyzer()
sw = set(get_stop_words("ru"))
exclude = set(string.punctuation)
c = 0

with codecs.open("chatbot/Otvety.txt", "r", "utf-8") as fin:
    for line in tqdm(fin):
        spls = preprocess_txt(line)
        sentences.append(spls)
        c += 1
        if c > 500000:
            break

0it [00:00, ?it/s]

In [5]:
# Обучим модель word2vec на наших вопросах
sentences = [i for i in sentences if len(i) > 2]
model = Word2Vec(sentences=sentences, vector_size=100, min_count=1, window=5)
model.save("w2v_model")

Теперь нам нужно сложить в индекс все вопросы. Используем библиотеку annoy. Проходимся по всем ответам, считаем, что вектор предложения - сумма word2vecов слов, которые входят в него (конечно же усредненная)

In [6]:
index = annoy.AnnoyIndex(100 ,'angular')

index_map = {}
counter = 0

with codecs.open("chatbot/prepared_answers.txt", "r", "utf-8") as f:
    for line in tqdm(f):
        n_w2v = 0
        spls = line.split("\t")
        index_map[counter] = spls[1]
        question = preprocess_txt(spls[0])
        vector = np.zeros(100)
        for word in question:
            if word in model.wv:
                vector += model.wv[word]
                n_w2v += 1
        if n_w2v > 0:
            vector = vector / n_w2v
        index.add_item(counter, vector)
            
        counter += 1

index.build(10)
index.save('speaker.ann')

0it [00:00, ?it/s]

True

Теперь остается реализовать метод, который получит на вход вопрос и найдет ответ к нему!
Мы препроцессим вопрос, находим ближайший вопрос и выбираем ответ на ближайший вопрос.

In [7]:
def find_answer(question):
    preprocessed_question = preprocess_txt(question)
    n_w2v = 0
    vector = np.zeros(100)
    for word in preprocessed_question:
        if word in model.wv:
            vector += model.wv[word]
            n_w2v += 1
    if n_w2v > 0:
        vector = vector / n_w2v
    answer_index = index.get_nns_by_vector(vector, 1)
    return index_map[answer_index[0]]

Проверка функции вызова ответа:

In [8]:
find_answer("Юбка детская ORBY")

'Сама всегда удивляюсь, неужели им мёдом там намазано? Я терпеть не могу такие взгляды, но всем всегда интересно и посмотреть на маленького малыша. Больше чем уверена, что таких людей самих раздражало, когда в коляску к их детям заглядывали. Один раз видела, как ребёнок в коляске в торговом центре был прикрыт белой пеленкой.. \n'

In [9]:
find_answer("Где ключи от танка")

'Наверное никак =(. \n'

# 9. Проект модуля

### Домашнее задание "PDA-1. Предобработка текстов с помощью Python":

Создать чат-бот-«барахольщик»: по продуктовому запросу он будет рекомендовать товары, по остальным запросам он будет отвечать «болталкой» (без фолбека).

### Задача:

- Файл с продуктами и описанием товаров расположен в файле "chatbot/ProductsDataset.csv"
- Обучить классификатор: продуктовый запрос vs. всё остальное (продуктовым можно считать запрос, который равен названию или описанию товара).
- Добавить логику поиска похожих товаров по продуктовому запросу.
- Вся логика должна быть завёрнута в метод get_answer(). Ответ на продуктовый запрос должен иметь вид "product_id title".

*Автотесты применяются к методу get_answer и проверяют несколько базовых сценариев:*

> *assert(get_answer(“Юбка детская ORBY”).startswith(“58e3cfe6132ca50e053f5f82”))*
> 
> *assert(not get_answer(“Где ключи от танка”).startswith(“5”))*

### Критерии проверки:

**→ Обучен классификатор «товарный запрос vs. болталка»**

- Осуществлён препроцессинг текста (как минимум удаление знаков препинания, приведение к нижнему регистру, стемминг/лемматизация).
- Текст векторизирован любым из изученных способов (CountVectorizer, TfidfVectorizer, HashingVectorizer, Word2Vec).
- Выборка разделена на обучающую и валидационную.
- Обучен классификатор с расчётом метрик на валидации (любое семейство алгоритмов, но предпочтительнее просто логистическая регрессия).
- Модель сохранена и при применении загружается из pkl-файла (или аналога).


**→ Реализован поиск похожих товаров в контентной части бота**

- Все названия товаров свёрнуты в векторное представление Word2Vec (предобученном или обученном на исходном датасете).
- Построен индекс по названиям документов.
- Для товарных запросов реализован поиск в индексе (запрос также оборачивается Word2Vec, происходит проход в индекс).


**→ Реализована болталка**

- Все вопросы из датасета свёрнуты Word2Vec в векторное представление.
- Построен индекс по вопросам.
- На запрос в болталку происходит поиск ближайшего вопроса и возвращается ответ на этот вопрос.

# Домашнее задание:

подгрузим данные о товарах:

In [10]:
products_df = pd.read_csv("chatbot/ProductsDataset.csv")
products_df.head(5)

,title,descrirption,product_id,category_id,subcategory_id,properties,image_links
0,Юбка детская ORBY,"Новая, не носили ни разу. В реале красивей чем...",58e3cfe6132ca50e053f5f82,22.0,2211,"{'detskie_razmer_rost': '81-86 (1,5 года)'}",http://cache3.youla.io/files/images/360_360/58...
1,Ботильоны,"Новые,привезены из Чехии ,указан размер 40,но ...",5667531b2b7f8d127d838c34,9.0,902,"{'zhenskaya_odezhda_tzvet': 'Зеленый', 'visota...",http://cache3.youla.io/files/images/360_360/5b...
2,Брюки,Размер 40-42. Брюки почти новые - не знаю как ...,59534826aaab284cba337e06,9.0,906,{'zhenskaya_odezhda_dzhinsy_bryuki_tip': 'Брюк...,http://cache3.youla.io/files/images/360_360/59...
3,Продам детские шапки,"Продам шапки,кажда 200р.Розовая и белая проданны.",57de544096ad842e26de8027,22.0,2217,"{'detskie_pol': 'Девочкам', 'detskaya_odezhda_...",http://cache3.youla.io/files/images/360_360/57...
4,Блузка,"Темно-синяя, 42 размер,состояние отличное,как ...",5ad4d2626c86cb168d212022,9.0,907,"{'zhenskaya_odezhda_tzvet': 'Синий', 'zhenskay...",http://cache3.youla.io/files/images/360_360/5a...


Сделает чат бот для ответов по товарам:

Подготовим исходные данные (диалоги и товары), переводим текст в вектор, чтобы потом обучать классификатор и делать поиск похожих вопросов/товаров:

In [11]:
from typing import Any, Dict, List, Tuple

ARTIFACT_DIR = "chatbot_artifacts"
INTENT_MODEL_PATH = os.path.join(ARTIFACT_DIR, "intent_model.pkl")
W2V_PATH = os.path.join(ARTIFACT_DIR, "w2v_bot.model")
CHAT_INDEX_PATH = os.path.join(ARTIFACT_DIR, "chat.ann")
CHAT_MAP_PATH = os.path.join(ARTIFACT_DIR, "chat_map.pkl")
PRODUCT_INDEX_PATH = os.path.join(ARTIFACT_DIR, "products.ann")
PRODUCT_MAP_PATH = os.path.join(ARTIFACT_DIR, "product_map.pkl")

# Функция для чтения файла диалогов Otvety.txt и собирает упрощенный файл prepared_answers.txt в формате: вопрос и ответ
def _prepare_answers_file() -> None:
    question: str | None = None
    written: bool = False
    with codecs.open("chatbot/prepared_answers.txt", "w", "utf-8") as fout:
        with codecs.open("chatbot/Otvety.txt", "r", "utf-8") as fin:
            for line in fin:
                is_separator = line.startswith("---")
                written = written and (not is_separator)
                question = question if (not is_separator) else None

                should_write = (not written) and (question is not None) and (not is_separator)
                fout.write((question.replace("\t", " ").strip() + "\t" + line.replace("\t", " ")) if should_write else "")
                written = written or should_write
                question = None if should_write else (line.strip() if ((not written) and (not is_separator)) else question)

# Функция для подготовки файла, читает prepared_answers.txt, каждую строку делит по первому табу на вопрос и ответ. Отбрасывает невалидные/пустые пары. Возвращает список пар вида (вопрос, ответ)
def _read_chat_pairs() -> List[Tuple[str, str]]:
    _prepare_answers_file()
    pairs: List[Tuple[str, str]] = []
    with codecs.open("chatbot/prepared_answers.txt", "r", "utf-8") as f:
        for line in f:
            parts = line.strip().split("\t", 1)
            valid = len(parts) == 2
            q = parts[0].strip() if valid else ""
            a = parts[1].strip() if valid else ""
            pairs.extend([(q, a)] * int(bool(q and a)))
    return pairs

# Загружаем таблицу товаров из ProductsDataset.csv в DataFrame. Промеряем что title и description не NaN и являются строками. "product_id" приводим к строке. Цель: чистый и стабильный DataFrame для обучения/поиска.
def _load_products() -> pd.DataFrame:
    df = pd.read_csv("chatbot/ProductsDataset.csv")
    df = df.rename(columns={"descrirption": "description"})
    df["title"] = df["title"].fillna("").astype(str)
    df["description"] = df["description"].fillna("").astype(str)
    df["product_id"] = df["product_id"].astype(str)
    return df

# Возьмем векторы слов из Word2Vec (только те слова, которые есть в словаре). Если слова найдены: возвращает средний вектор (mean pooling), тип float32. Если не найдено ни одного слова: возвращает нулевой вектор нужной размерности. Базовая функция, чтобы превратить текст в один числовой вектор для индексации и поиска ближайших соседей.
def _sentence_vector(tokens: List[str], w2v_model: Word2Vec) -> np.ndarray:
    vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]
    base = np.zeros(w2v_model.vector_size, dtype=np.float32)
    return np.mean(vectors, axis=0).astype(np.float32) if len(vectors) else base

Реализуем схему train once -> save -> fast load on next runs, с авто-восстановлением, если артефакты отсутствуют или повреждены.

In [12]:
# Функция обучает и сохраняет все артефакты. Создает папку для файлов моделей и загружаем данные: пары вопрос-ответ для болталки и таблицу товаров.
# Готовит датасет для классификатора намерения: Положительный класс: названия и описания товаро; Отрицательный класс: вопросы из болталки.
# Текст через preprocess_txt, делим на train/val/test, векторизует TF-IDF (1-2 граммы), обучает LogisticRegression.
# Сохраняем классификатор и векторизатор в intent_model.pkl.
# Обучаем Word2Vec на объединенном корпусе (вопросы, title, description) и сохраняем модель.
def _build_and_save_artifacts() -> None:
    os.makedirs(ARTIFACT_DIR, exist_ok=True)

    chat_pairs = _read_chat_pairs()
    products = _load_products()

    positive_texts = products["title"].tolist() + products["description"].tolist()
    negative_texts = [q for q, _ in chat_pairs]

    x_raw = positive_texts + negative_texts
    y = np.array([1] * len(positive_texts) + [0] * len(negative_texts))
    x_clean = [" ".join(preprocess_txt(text)) for text in x_raw]

    x_train_raw, x_holdout_raw, y_train, y_holdout = train_test_split(
        x_clean,
        y,
        test_size=0.3,
        random_state=42,
        stratify=y
    )
    x_val_raw, x_test_raw, y_val, y_test = train_test_split(
        x_holdout_raw,
        y_holdout,
        test_size=0.5,
        random_state=42,
        stratify=y_holdout
    )

    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
    x_train = vectorizer.fit_transform(x_train_raw)
    x_val = vectorizer.transform(x_val_raw)
    x_test = vectorizer.transform(x_test_raw)

    clf = LogisticRegression(max_iter=1000, class_weight="balanced")
    clf.fit(x_train, y_train)

    y_val_pred = clf.predict(x_val)
    y_test_pred = clf.predict(x_test)

    val_acc = accuracy_score(y_val, y_val_pred)
    test_acc = accuracy_score(y_test, y_test_pred)

    print(f"Validation Accuracy: {val_acc:.4f}")
    print(classification_report(y_val, y_val_pred, digits=4))
    print(f"Test Accuracy: {test_acc:.4f}")
    print(classification_report(y_test, y_test_pred, digits=4))

    with open(INTENT_MODEL_PATH, "wb") as f:
        pickle.dump({"vectorizer": vectorizer, "clf": clf}, f)

    corpus = ([preprocess_txt(q) for q, _ in chat_pairs]
              + [preprocess_txt(t) for t in products["title"].tolist()]
              + [preprocess_txt(d) for d in products["description"].tolist()])

    w2v = Word2Vec(
        sentences=corpus,
        vector_size=100,
        window=5,
        min_count=1,
        sg=1,
        workers=4,
        epochs=15
    )
    w2v.save(W2V_PATH)

    chat_index = annoy.AnnoyIndex(w2v.vector_size, "angular")
    chat_map: Dict[int, str] = {}
    for idx, (question, answer) in enumerate(chat_pairs):
        chat_index.add_item(idx, _sentence_vector(preprocess_txt(question), w2v))
        chat_map[idx] = answer
    chat_index.build(20)
    chat_index.save(CHAT_INDEX_PATH)
    with open(CHAT_MAP_PATH, "wb") as f:
        pickle.dump(chat_map, f)

    product_index = annoy.AnnoyIndex(w2v.vector_size, "angular")
    product_map: Dict[int, Dict[str, str]] = {}
    exact_title_map: Dict[str, int] = {}

    for idx, row in products.reset_index(drop=True).iterrows():
        title = str(row["title"])
        title_norm = " ".join(preprocess_txt(title))
        product_index.add_item(idx, _sentence_vector(preprocess_txt(title), w2v))
        product_map[idx] = {"product_id": row["product_id"], "title": title}
        if title_norm:
            exact_title_map[title_norm] = idx

    product_index.build(20)
    product_index.save(PRODUCT_INDEX_PATH)
    with open(PRODUCT_MAP_PATH, "wb") as f:
        pickle.dump({"product_map": product_map, "exact_title_map": exact_title_map}, f)


# Загружаем все готовые артефакты с диска. Загружаем классификатор и TF-IDF, Word2Vec, индекс болталки и карту ответов, индекс товаров и карты товаров/точных заголовков.
# Возвращает единый словарь bot со всем необходимым для ответа на запрос.
def _load_bot() -> Dict[str, Any]:
    with open(INTENT_MODEL_PATH, "rb") as f:
        intent_data = pickle.load(f)

    w2v = Word2Vec.load(W2V_PATH)

    chat_index = annoy.AnnoyIndex(w2v.vector_size, "angular")
    chat_index.load(CHAT_INDEX_PATH)
    with open(CHAT_MAP_PATH, "rb") as f:
        chat_map = pickle.load(f)

    product_index = annoy.AnnoyIndex(w2v.vector_size, "angular")
    product_index.load(PRODUCT_INDEX_PATH)
    with open(PRODUCT_MAP_PATH, "rb") as f:
        product_data = pickle.load(f)

    return {
        "vectorizer": intent_data["vectorizer"],
        "clf": intent_data["clf"],
        "w2v": w2v,
        "chat_index": chat_index,
        "chat_map": chat_map,
        "product_index": product_index,
        "product_map": product_data["product_map"],
        "exact_title_map": product_data["exact_title_map"],
    }


# Делаем инициализацию, сначала пробуем просто загрузить готовые файлы.
# Если файлов нет, они битые или структура не та: Переобучает и пересохраняет все через _build_and_save_artifacts и снова загружаем
def _get_bot() -> Dict[str, Any]:
    try:
        return _load_bot()
    except (FileNotFoundError, OSError, EOFError, pickle.UnpicklingError, KeyError):
        _build_and_save_artifacts()
        return _load_bot()

In [13]:
# Нормализуем вопрос через preprocess_txt. Превращаем текст в TF-IDF-вектор через bot["vectorizer"].
# Классификатор bot["clf"] предсказывает класс: True (1) -> это товарный запрос. False (0) -> это обычный вопрос для болталки

def _predict_is_product(question: str, bot: Dict[str, Any]) -> bool:
    question_norm = " ".join(preprocess_txt(question))
    x = bot["vectorizer"].transform([question_norm])
    return bool(bot["clf"].predict(x)[0])

# Токенизируем и нормализуем вопрос. Строим вектор запроса через Word2Vec (_sentence_vector).
# Ищем ближайший товар в Annoy-индексе bot["product_index"]. Если есть точное совпадение нормализованного title в exact_title_map, берет его (это приоритетнее nearest neighbor).
def _answer_product(question: str, bot: Dict[str, Any]) -> str:
    tokens = preprocess_txt(question)
    question_norm = " ".join(tokens)
    nearest_idx = bot["product_index"].get_nns_by_vector(_sentence_vector(tokens, bot["w2v"]), 1)[0]
    idx = bot["exact_title_map"].get(question_norm, nearest_idx)
    item = bot["product_map"][idx]
    return f"{item['product_id']} {item['title']}"

# Векторизуем вопрос через Word2Vec. Ищем ближайший вопрос в chat-индексе bot["chat_index"]. Возвращаем соответствующий ответ из bot["chat_map"].
def _answer_chat(question: str, bot: Dict[str, Any]) -> str:
    vec = _sentence_vector(preprocess_txt(question), bot["w2v"])
    idx = bot["chat_index"].get_nns_by_vector(vec, 1)[0]
    return bot["chat_map"][idx]

# По результату выбираем функцию:
# 0 -> _answer_chat
# 1 -> _answer_product
# Возвращает итоговый ответ.
def get_answer(question: str) -> str:
    fn = (_answer_chat, _answer_product)[int(_predict_is_product(question, BOT))]
    return fn(question, BOT)


_build_and_save_artifacts()
BOT = _load_bot()

Validation Accuracy: 0.9848
              precision    recall  f1-score   support

           0     0.9962    0.9877    0.9919    174499
           1     0.8235    0.9376    0.8768     10665

    accuracy                         0.9848    185164
   macro avg     0.9098    0.9626    0.9344    185164
weighted avg     0.9862    0.9848    0.9853    185164

Test Accuracy: 0.9849
              precision    recall  f1-score   support

           0     0.9958    0.9881    0.9920    174500
           1     0.8273    0.9324    0.8767     10664

    accuracy                         0.9849    185164
   macro avg     0.9116    0.9602    0.9343    185164
weighted avg     0.9861    0.9849    0.9853    185164



### Проверка:

Вопрос №1

In [14]:
get_answer("Юбка детская ORBY")

'58e3cfe6132ca50e053f5f82 Юбка детская ORBY'

Вопрос №2

In [ ]:
get_answer("Где ключи от танка")

'У тебя в сама знаешь где.'

Вопрос №3

In [19]:
get_answer("нужна детская одежда")

'5ae1b9fd2c593e8ea11768a4 Детская одежда 3-4 года'

Вопрос №4

In [20]:
get_answer("какая сегодня погода?")

'Верхняя Волга - жара, дым, листопад, трава желтеет....'

# Вывод по проделанной работе:

В ходе работы были последовательно выполнены основные этапы проекта: подготовка данных, анализ, разработка и проверка решения. Произведена проверка отвутов чат бота.